# 07 — 5/10/15-minute coverage


In [1]:
from pathlib import Path
import sys

import pandas as pd

# ---------------------------------------------------------
# Resolve project root
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/subhankarbiswas/smart-city-knowledge-graph-v3


In [2]:
from src.network_accessibility import threshold_coverage

print("threshold_coverage imported successfully.")

threshold_coverage imported successfully.


In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ACCESSIBILITY_PATH = (
    PROCESSED_DIR / "network_accessibility.csv"
)

COVERAGE_PATH = (
    PROCESSED_DIR / "accessibility_coverage.csv"
)

SUMMARY_PATH = (
    PROCESSED_DIR / "accessibility_coverage_summary.csv"
)

if not ACCESSIBILITY_PATH.exists():
    raise FileNotFoundError(
        "Network accessibility results were not found:\n"
        f"{ACCESSIBILITY_PATH}\n\n"
        "Run Notebook 06 first."
    )

print(f"Input:  {ACCESSIBILITY_PATH}")
print(f"Output: {COVERAGE_PATH}")

Input:  /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/network_accessibility.csv
Output: /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_coverage.csv


In [4]:
df = pd.read_csv(ACCESSIBILITY_PATH)

print("=" * 60)
print("NETWORK ACCESSIBILITY DATA")
print("=" * 60)

print(f"Rows:    {len(df):,}")
print(f"Columns: {len(df.columns):,}")

print("\nColumns:")
for column in df.columns:
    print(f"  - {column}")

NETWORK ACCESSIBILITY DATA
Rows:    110
Columns: 5

Columns:
  - origin_index
  - network_node
  - travel_time_s
  - travel_time_min
  - service


In [5]:
if df.empty:
    raise ValueError(
        "network_accessibility.csv is empty. "
        "Run Notebook 06 and verify that accessibility "
        "calculations produced results."
    )

if "service" not in df.columns:
    raise KeyError(
        "Required column 'service' is missing from "
        "network_accessibility.csv."
    )

df["service"] = (
    df["service"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

if not df["service"].astype(bool).any():
    raise ValueError(
        "The 'service' column does not contain valid "
        "service types."
    )

print("Dataset validation passed.")

Dataset validation passed.


In [6]:
service_counts = (
    df["service"]
    .value_counts()
    .rename_axis("service")
    .reset_index(name="records")
)

print("=" * 60)
print("ACCESSIBILITY RECORDS BY SERVICE")
print("=" * 60)

display(service_counts)

ACCESSIBILITY RECORDS BY SERVICE


,service,records
0,hospital,22
1,pharmacy,22
2,supermarket,22
3,train_station,22
4,library,22


In [7]:
print("=" * 60)
print("ACCESSIBILITY DATA SAMPLE")
print("=" * 60)

display(df.head(10))

ACCESSIBILITY DATA SAMPLE


,origin_index,network_node,travel_time_s,travel_time_min,service
0,6,5144049364,1933.581874,32.226365,hospital
1,7,428473989,2343.797150,39.063286,hospital
2,8,428469705,2151.900526,35.865009,hospital
3,9,428473989,2343.797150,39.063286,hospital
4,10,254161663,715.177882,11.919631,hospital
5,11,28923280,596.674421,9.944574,hospital
6,12,1898284092,1063.308010,17.721800,hospital
7,13,2181649761,2176.655892,36.277598,hospital
8,14,1248399607,1668.147245,27.802454,hospital
9,15,2846150385,1204.700073,20.078335,hospital


In [8]:
print("=" * 60)
print("NUMERIC COLUMNS")
print("=" * 60)

numeric_columns = df.select_dtypes(
    include="number"
).columns.tolist()

print(numeric_columns)

NUMERIC COLUMNS
['origin_index', 'network_node', 'travel_time_s', 'travel_time_min']


In [9]:
parts = []

print("=" * 60)
print("ACCESSIBILITY COVERAGE")
print("=" * 60)

for service, part in df.groupby("service", sort=True):

    print(f"\nService: {service}")
    print("-" * 40)
    print(f"Input records: {len(part):,}")

    if part.empty:
        print("No records — skipping.")
        continue

    try:
        coverage_part = threshold_coverage(
            part.copy()
        )

        if coverage_part is None:
            print("threshold_coverage returned None — skipping.")
            continue

        if coverage_part.empty:
            print("No coverage results returned.")
            continue

        coverage_part = coverage_part.copy()
        coverage_part["service"] = service

        parts.append(coverage_part)

        print(
            f"Coverage records: "
            f"{len(coverage_part):,}"
        )

    except Exception as exc:
        print(
            f"Coverage calculation failed for "
            f"{service}: {exc}"
        )

ACCESSIBILITY COVERAGE

Service: hospital
----------------------------------------
Input records: 22
Coverage records: 3

Service: library
----------------------------------------
Input records: 22
Coverage records: 3

Service: pharmacy
----------------------------------------
Input records: 22
Coverage records: 3

Service: supermarket
----------------------------------------
Input records: 22
Coverage records: 3

Service: train_station
----------------------------------------
Input records: 22
Coverage records: 3


In [10]:
if parts:
    coverage = pd.concat(
        parts,
        ignore_index=True
    )
else:
    coverage = pd.DataFrame()

print("=" * 60)
print("COMBINED COVERAGE RESULTS")
print("=" * 60)

print(f"Rows: {len(coverage):,}")

if not coverage.empty:
    display(coverage.head(10))
else:
    print("No coverage results were generated.")

COMBINED COVERAGE RESULTS
Rows: 15


,threshold_min,origins,reached,coverage_pct,service
0,5,22,0,0.000000,hospital
1,10,22,3,13.636364,hospital
2,15,22,7,31.818182,hospital
3,5,22,0,0.000000,library
4,10,22,5,22.727273,library
5,15,22,9,40.909091,library
6,5,22,6,27.272727,pharmacy
7,10,22,10,45.454545,pharmacy
8,15,22,13,59.090909,pharmacy
9,5,22,12,54.545455,supermarket


In [11]:
if not coverage.empty:
    
    print("Coverage columns:")
    for column in coverage.columns:
        print(f"  - {column}")

    print("\nData types:")
    display(
        coverage.dtypes
        .to_frame("dtype")
    )

Coverage columns:
  - threshold_min
  - origins
  - reached
  - coverage_pct
  - service

Data types:


,dtype
threshold_min,int64
origins,int64
reached,int64
coverage_pct,float64
service,str


In [12]:
if not coverage.empty:
    
    coverage_summary = (
        coverage
        .groupby("service")
        .size()
        .reset_index(name="coverage_records")
        .sort_values(
            "coverage_records",
            ascending=False
        )
    )

    print("=" * 60)
    print("COVERAGE SUMMARY BY SERVICE")
    print("=" * 60)

    display(coverage_summary)

COVERAGE SUMMARY BY SERVICE


,service,coverage_records
0,hospital,3
1,library,3
2,pharmacy,3
3,supermarket,3
4,train_station,3


In [13]:
if not coverage.empty:
    
    threshold_columns = [
        column
        for column in coverage.columns
        if "threshold" in column.lower()
        or "minute" in column.lower()
        or "coverage" in column.lower()
    ]

    print("=" * 60)
    print("COVERAGE-RELATED COLUMNS")
    print("=" * 60)

    if threshold_columns:
        for column in threshold_columns:
            print(f"  - {column}")
    else:
        print("No automatically detected coverage columns.")

COVERAGE-RELATED COLUMNS
  - threshold_min
  - coverage_pct


In [14]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

coverage.to_csv(
    COVERAGE_PATH,
    index=False
)

print(f"Coverage results saved to:")
print(COVERAGE_PATH)

Coverage results saved to:
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_coverage.csv


In [15]:
if not coverage.empty:
    
    coverage_summary = (
        coverage
        .groupby("service")
        .size()
        .reset_index(name="coverage_records")
        .sort_values(
            "coverage_records",
            ascending=False
        )
    )

    coverage_summary.to_csv(
        SUMMARY_PATH,
        index=False
    )

    print(f"Coverage summary saved to:")
    print(SUMMARY_PATH)

Coverage summary saved to:
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_coverage_summary.csv


In [16]:
print("=" * 60)
print("NOTEBOOK 07 VALIDATION")
print("=" * 60)

print(f"Input accessibility records: {len(df):,}")
print(f"Coverage records:             {len(coverage):,}")

print("\nServices processed:")

if not coverage.empty:
    processed_services = (
        coverage["service"]
        .dropna()
        .unique()
        .tolist()
    )

    for service in sorted(processed_services):
        count = (
            coverage["service"]
            .eq(service)
            .sum()
        )
        print(f"  {service:<15} {count:,}")
else:
    print("  No services processed.")

print("\nOutput files:")

for path in [
    COVERAGE_PATH,
    SUMMARY_PATH,
]:
    if path.exists():
        print(f"  ✓ {path}")
    else:
        print(f"  ✗ {path}")

NOTEBOOK 07 VALIDATION
Input accessibility records: 110
Coverage records:             15

Services processed:
  hospital        3
  library         3
  pharmacy        3
  supermarket     3
  train_station   3

Output files:
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_coverage.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_coverage_summary.csv
